# TP - Extracción de texto y datos de un PDF

PDF usado: *SqueezeNet: AlexNet-level accuracy with 50x fewer parameters and <0.5MB model size* (arXiv:1602.07360)

In [ ]:
!pip install -q pymupdf pdfplumber nltk

In [ ]:
import re
from collections import Counter

import requests
import fitz
import pdfplumber
import pandas as pd
import matplotlib.pyplot as plt
import nltk

## 1. Adquisición y metadatos

In [ ]:
url = "https://arxiv.org/pdf/1602.07360"
archivo = "squeezenet.pdf"

# descargo el pdf y lo guardo
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
r.raise_for_status()
with open(archivo, "wb") as f:
    f.write(r.content)

print("Descargado:", archivo, "-", len(r.content) // 1024, "KB")

In [ ]:
doc = fitz.open(archivo)
meta = doc.metadata

# si el campo viene vacio lo aclaro
def ver(campo):
    return meta.get(campo) or "(vacío)"

print("Título:", ver("title"))
print("Autor:", ver("author"))
print("Páginas:", doc.page_count)
print("Creator:", ver("creator"))
print("Producer:", ver("producer"))

In [ ]:
# chequeo que todas las paginas tengan texto (que no sea escaneado)
for i, pag in enumerate(doc):
    print("Página", i + 1, "->", len(pag.get_text()), "caracteres")

**Conclusión punto 1:**

El PDF se descargó con `requests` y se validó con la respuesta HTTP (`raise_for_status`). Los metadatos se leyeron con `doc.metadata` de PyMuPDF. Usé PyMuPDF porque devuelve todos los campos en un diccionario y además da el número de páginas con `page_count`. En los papers de arXiv es común que el título o el autor vengan vacíos, porque dependen de cómo el autor configuró LaTeX. El creator/producer muestra que el PDF fue generado con LaTeX (pdfTeX), es decir que es un PDF "nativo" y no escaneado. Esto se confirma porque todas las páginas devuelven texto.

## 2. Análisis estructural del texto

Para extraer el texto uso `page.get_text("text")` de PyMuPDF. Con `repr()` se ven los caracteres especiales tal cual vienen.

In [ ]:
textos = [pag.get_text("text") for pag in doc]
texto = "".join(textos)

# texto crudo de la primera pagina
print(repr(textos[0][:1500]))

In [ ]:
# lo mismo pero impreso normal para comparar
print(textos[0][:1500])

In [ ]:
# bloques de la pagina 1: cada bloque es mas o menos un parrafo o un titulo
for b in doc[0].get_text("blocks")[:10]:
    print(b[5], repr(b[4][:80]))

In [ ]:
# cuento los caracteres que podrian ser separadores
for nombre, c in [("\\n", "\n"), ("\\r", "\r"), ("\\t", "\t"), ("\\xa0", "\xa0"), ("\\x0c", "\x0c")]:
    print(nombre, texto.count(c))

print("lineas en blanco (\\n\\n):", len(re.findall(r"\n\s*\n", texto)))
print("palabras cortadas con guion (-\\n):", len(re.findall(r"\w-\n\w", texto)))
print("2 o mas espacios seguidos:", len(re.findall(r" {2,}", texto)))

In [ ]:
# busco los titulos de seccion (ej: "1 INTRODUCTION", "3.1 ...")
secciones = re.findall(r"^(\d+(?:\.\d+)*)\.?\s+([A-Z][A-Za-z ,:\-]{2,60})$", texto, re.MULTILINE)
for s in secciones:
    print(s)

**Respuesta: ¿qué caracteres funcionan como separadores en este archivo?**

| Carácter | ¿Aparece? | Función en este PDF |
|---|---|---|
| `\n` (salto de línea) | Sí, es el único separador | Marca el final de cada **línea visual** del PDF |
| `\n` + línea con número y título | Sí | Marca el inicio de una **sección** (ej. `1 INTRODUCTION`) |
| `-\n` (guion + salto) | Sí | Palabra cortada entre dos líneas, **no** es un separador real |
| `\r` (retorno de carro) | No | - |
| `\t` (tabulación) | No | - |
| `\n\n` (línea en blanco) | No | Los párrafos **no** se separan con línea en blanco |
| Varios espacios seguidos | Casi no | Solo en alguna fórmula o tabla |

(verificar los valores con la salida de las celdas de conteo)

**Justificación:**

- **Líneas:** el conteo muestra muchos `\n` y ningún `\r` ni `\t`. PyMuPDF pone un `\n` al final de cada renglón, y eso se ve en el `repr()`: las oraciones quedan partidas en varias líneas de largo parecido (el ancho de la columna). O sea que `\n` no indica fin de oración ni de párrafo.
- **Párrafos:** como no hay líneas en blanco, no hay un carácter que separe párrafos. La única forma de detectarlos es por posición: con `get_text("blocks")` cada bloque es un párrafo, porque PyMuPDF los agrupa por coordenadas.
- **Secciones:** tampoco tienen un carácter especial. Se reconocen por el patrón número + título en una línea propia, que es lo que encuentra la expresión regular.
- **Capítulos:** el documento es un paper y no tiene capítulos. Las páginas se extraen por separado y no hay un `\x0c` (salto de página) entre ellas.

**Conclusión punto 2:**

En un PDF el texto no guarda la estructura con caracteres como un archivo de texto. Lo que existe es la posición de cada palabra en la página, y la librería reconstruye las líneas a partir de eso. Por eso el único separador real es `\n` (por línea) y para recuperar párrafos y secciones hay que usar los bloques o patrones como el número del título. Antes de analizar el texto conviene unir las palabras cortadas con guion y reemplazar los `\n` por espacios.

## 3. Extracción de tablas

In [ ]:
# recorro las paginas buscando tablas con pdfplumber
# pruebo tambien con la estrategia "text" porque las tablas de los papers casi no tienen lineas verticales
config = {"vertical_strategy": "text", "horizontal_strategy": "lines"}

encontradas = []
with pdfplumber.open(archivo) as pdf:
    for i, pag in enumerate(pdf.pages):
        for t in pag.extract_tables() + pag.extract_tables(config):
            if len(t) >= 3 and len(t[0]) >= 2:
                encontradas.append((i + 1, len(t), len(t[0]), t))

for p, filas, cols, _ in encontradas:
    print("Página", p, "-", filas, "filas x", cols, "columnas")

In [ ]:
# me quedo con la tabla mas grande
pagina_tabla, _, _, tabla = max(encontradas, key=lambda x: x[1] * x[2])
print("Página:", pagina_tabla)

df = pd.DataFrame(tabla)
df

In [ ]:
# limpieza: saco saltos de linea, paso vacios a NaN y borro filas/columnas vacias
df = df.map(lambda x: " ".join(str(x).split()) if x is not None else None)
df = df.replace("", None)
df = df.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)

# uso la primera fila como nombres de columnas
df.columns = [c if c else f"col_{i}" for i, c in enumerate(df.iloc[0])]
df = df[1:].reset_index(drop=True)
df

**Conclusión punto 3:**

Para las tablas usé `pdfplumber` porque permite configurar cómo detecta filas y columnas. Con la configuración por defecto (`lines`) solo encuentra tablas con todas las líneas dibujadas. En los papers las tablas casi no tienen líneas verticales, por eso también probé `vertical_strategy="text"`, que arma las columnas según la alineación del texto. Elegí la tabla más grande encontrada.

La tabla sale como una lista de listas con celdas `None`, saltos de línea dentro de las celdas y filas vacías. La limpieza consistió en sacar los saltos de línea y espacios de más, pasar las celdas vacías a `NaN`, borrar filas y columnas vacías y usar la primera fila como nombre de columnas. Algunos encabezados pueden quedar cortados o mezclados, porque en el PDF ocupan dos renglones; en ese caso hay que renombrarlos a mano.

## 4. Análisis de frecuencias

Elijo una página que tenga solo texto (sin tablas ni figuras).

In [ ]:
# palabras por pagina, para elegir una con mucho texto
paginas_tablas = {e[0] for e in encontradas}
for i, t in enumerate(textos):
    tiene_img = len(doc[i].get_images()) > 0
    print(i + 1, len(t.split()), "palabras", "| tabla" if i + 1 in paginas_tablas else "", "| imagen" if tiene_img else "")

In [ ]:
# elijo la pagina mirando la salida de arriba (mucho texto, sin tabla ni imagen)
pagina = 2
texto_pag = textos[pagina - 1]
print(texto_pag[:1000])

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# uno palabras cortadas con guion, saco los saltos de linea y paso a minusculas
t = re.sub(r"(\w)-\n(\w)", r"\1\2", texto_pag)
t = t.replace("\n", " ").lower()

tokens = word_tokenize(t)
print(len(tokens), "tokens")
print(tokens[:30])

In [ ]:
stop = set(stopwords.words("english"))

# me quedo solo con palabras (sin puntuacion ni numeros) que no sean stop words
limpios = [w for w in tokens if w.isalpha() and w not in stop]
print(len(limpios), "tokens despues de limpiar")

In [ ]:
top = Counter(limpios).most_common(15)
palabras, cant = zip(*top)

plt.figure(figsize=(10, 5))
plt.bar(palabras, cant)
plt.xticks(rotation=45)
plt.title(f"15 palabras más frecuentes - página {pagina}")
plt.ylabel("frecuencia")
plt.tight_layout()
plt.show()

**¿Hay tokens que también consideraría stop words?**

(completar con las palabras que salen en el gráfico)

Sí. Aparecen palabras que no están en la lista de NLTK pero que no dicen nada del contenido de la página, por ejemplo restos de citas como "et" y "al", y palabras muy comunes en cualquier paper como "also", "using" o "show". También hay palabras que aparecen en todo el documento (como "cnn" o "model"), que para analizar esta página en particular no aportan mucho.

In [ ]:
# pruebo sacando esas palabras tambien
stop_extra = {"et", "al", "also", "using", "use", "used", "show", "however", "e", "g", "i"}
limpios2 = [w for w in limpios if w not in stop_extra]

top2 = Counter(limpios2).most_common(15)
palabras2, cant2 = zip(*top2)

plt.figure(figsize=(10, 5))
plt.bar(palabras2, cant2, color="orange")
plt.xticks(rotation=45)
plt.title(f"Top 15 sin stop words extra - página {pagina}")
plt.tight_layout()
plt.show()

**Conclusión punto 4:**

La tokenización con `word_tokenize` separa las palabras de los signos de puntuación, por eso después se filtraron con `isalpha()`, que también saca números y referencias como `[12]`. Antes de tokenizar fue necesario unir las palabras cortadas con guion y sacar los `\n` (lo visto en el punto 2), porque si no aparecían tokens rotos.

La lista de stop words de NLTK saca las palabras funcionales del inglés, pero no las propias de un paper científico. Al sacar también esas palabras, el gráfico muestra mejor de qué trata la página. Esto muestra que la lista de stop words depende del tipo de texto que se analiza y conviene ampliarla para cada caso.

## Fuentes utilizadas

- Iandola, F. N., Han, S., Moskewicz, M. W., Ashraf, K., Dally, W. J. y Keutzer, K. (2016). *SqueezeNet: AlexNet-level accuracy with 50x fewer parameters and <0.5MB model size*. arXiv:1602.07360. https://arxiv.org/abs/1602.07360
- Documentación de Requests: https://requests.readthedocs.io
- Documentación de PyMuPDF: https://pymupdf.readthedocs.io
- pdfplumber (repositorio y documentación): https://github.com/jsvine/pdfplumber
- Documentación de pandas: https://pandas.pydata.org/docs/
- NLTK - Natural Language Toolkit: https://www.nltk.org
- Bird, S., Klein, E. y Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly. https://www.nltk.org/book/
- Documentación de Matplotlib: https://matplotlib.org/stable/